# Data Preparation (by Merah Putih)

## 1. Prepare ATD Data

### 1.1. Convert Tactical ATD to CSV

This converts the raw tactical ATD data (the metadata XML and raw tracking text) into a CSV.
It writes `data/tactical/BAR-ATH_tactical_ds.csv`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from kloppy import metrica

import games

import time
checkpoint_time = time.time()

paths = games.asset_paths(source='tactical')
dataset = metrica.load_tracking_epts(meta_data=paths['atd_meta'],
                                     raw_data=paths['atd_raw'])
frame_df = dataset.to_df()
frame_df = frame_df.rename(columns={'frame_id': 'video_frame'})
frame_df['video_frame'] = frame_df['video_frame'].astype(int) - 1
frame_df.loc[frame_df['video_frame'] >= 110808, 'period_id'] = 2
print('Duration point 1: %.2f seconds' % (time.time() - checkpoint_time))
checkpoint_time = time.time()

csv_path = os.path.join(games.DATA_DIR, 'tactical', 'BAR-ATH_tactical_ds.csv')

ball_speed_by_frame = {}
with open(paths['atd_raw']) as f:
    for line in f:
        atd_frame, _players, ball = line.rstrip('\r\n').split(':')
        speed = ball.split(',')[2]
        ball_speed_by_frame[int(atd_frame)] = (
            float(speed) if speed != 'NaN' else np.nan
        )
print('Duration point 2: %.2f seconds' % (time.time() - checkpoint_time))
checkpoint_time = time.time()

frame_df['ball_speed'] = (
    (frame_df['video_frame'].astype(int) + 1).map(ball_speed_by_frame)
)

# Add per-player positions in metres, distance to the ball, and team name.
length = dataset.metadata.pitch_dimensions.pitch_length
width = dataset.metadata.pitch_dimensions.pitch_width
ball_x_m = frame_df['ball_x'] * length
ball_y_m = frame_df['ball_y'] * width

extra = {}
for player in dataset.metadata.teams[0].players + dataset.metadata.teams[1].players:
    pid = player.player_id
    x_m = frame_df['%s_x' % pid] * length
    y_m = frame_df['%s_y' % pid] * width
    extra['%s_x_m' % pid] = x_m
    extra['%s_y_m' % pid] = y_m
    extra['%s_ball_dist_m' % pid] = np.hypot(x_m - ball_x_m, y_m - ball_y_m)
    extra['%s_team' % pid] = player.team.name

frame_df = pd.concat([frame_df, pd.DataFrame(extra, index=frame_df.index)], axis=1)
frame_df.to_csv(csv_path, index=False)
print('Duration point 3: %.2f seconds' % (time.time() - checkpoint_time))
print('Wrote %d frames to %s' % (len(frame_df), csv_path))


Duration point 1: 119.96 seconds
Duration point 2: 0.76 seconds
Duration point 3: 49.38 seconds
Wrote 199319 frames to /Users/hafizbadrielubis/Developments/Hubble/more-than-a-hack-2027/data/tactical/BAR-ATH_tactical_ds.csv


### 1.2. ATD Data Sampling

Take sample of 5 minutes worth of frame from the CSV file created from step 1 (from frame 111000 to 18600).

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd().resolve().parent / "data" / "tactical"
SRC = DATA_DIR / "BAR-ATH_tactical_ds.csv"
DST = DATA_DIR / "BAR-ATH_tactical_sample.csv"
# START_FRAME, END_FRAME = 11100, 18600
START_FRAME, END_FRAME = 11185, 18684

df = pd.read_csv(SRC)
sample = df[df["video_frame"].between(START_FRAME, END_FRAME)]
sample.to_csv(DST, index=False)
print(f"Wrote {len(sample)} rows to {DST}")

Wrote 7500 rows to /Users/hafizbadrielubis/Developments/Hubble/more-than-a-hack-2027/data/tactical/BAR-ATH_tactical_sample.csv


# 2. Prepare Smart Tagging Data

### 2.1. Convert Smart Tagging to CSV

This converts the raw smart tagging data (the metadata XML and raw tracking text) into a CSV.
It writes `data/tactical/BAR-ATH_smart_tagging_ds.csv`.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from kloppy import sportscode

import games

FPS = games.GAMES[games.GAME]['fps']
path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
events['start_frame'] = (events['timestamp'].dt.total_seconds() * FPS).round().astype(int)
events['end_frame'] = (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
events.to_csv(os.path.join(games.DATA_DIR, 'tactical', 'BAR-ATH_smart_tagging_ds.csv'), index=False)
print('%d instances' % len(events))

events.head()


638 instances


,code_id,period_id,timestamp,end_timestamp,code,Team,Type,Half,Max Players in the box,Side,Direction of ball entry,start_frame,end_frame
0,1,1,0 days 00:07:23.880000,0 days 00:07:28.880000,SET PIECES,N/A,Kick Off (Start),1st Half,None,NaN,NaN,11097,11222
1,2,1,0 days 00:33:05.240000,0 days 00:33:10.240000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN,49631,49756
2,3,1,0 days 00:41:20.480000,0 days 00:41:25.480000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN,62012,62137
3,4,1,0 days 00:50:18.280000,0 days 00:50:23.280000,SET PIECES,N/A,Kick Off (After Goal),1st Half,None,NaN,NaN,75457,75582
4,5,1,0 days 01:15:46.400000,0 days 01:15:51.400000,SET PIECES,N/A,Kick Off (Start),2nd Half,None,NaN,NaN,113660,113785


### 2.2. Smart Tagging Data Sampling

Take sample of 5 minutes worth of frame from the CSV file created from step 1 (from frame 111000 to 18600).

In [3]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd().resolve().parent / "data" / "tactical"
SRC = DATA_DIR / "BAR-ATH_smart_tagging_ds.csv"
DST = DATA_DIR / "BAR-ATH_tactical_smart_tagging_sample.csv"
# START_FRAME, END_FRAME = 11100, 18600
START_FRAME, END_FRAME = 11185, 18684

df = pd.read_csv(SRC)
# Keep events whose [start_frame, end_frame] range overlaps [START_FRAME, END_FRAME]
sample = df[(df["start_frame"] <= END_FRAME) & (df["end_frame"] >= START_FRAME)]
sample.to_csv(DST, index=False)
print(f"Wrote {len(sample)} rows to {DST}")

Wrote 30 rows to /Users/hafizbadrielubis/Developments/Hubble/more-than-a-hack-2027/data/tactical/BAR-ATH_tactical_smart_tagging_sample.csv
